## Generación de datos sintéticos

Se utiliza la librería `Faker` con apoyo de la librería `random` para la generación de un set de datos sintéticos realistas con al menos.
  - 500 clientes
  - 70 productos
  - 2000 pedidos
  - ~4500 líneas de pedido (media 2-3 productos por pedido)
  - Pagos correspondientes a cada pedido
  - Valoraciones para ~35% de los productos entregados \

Se procura que los datos tengan sentido de negocio con precios coherentes, fechas ordenadas, etc.

In [2]:
# Imoportar módulos
import random
from faker import Faker
import pandas as pd
from datetime import datetime, timedelta

In [3]:
# Instanciamos Faker con localización en español
fake = Faker('es_ES')

# Fijamos semilla para reproducibilidad (opcional pero recomendado)
Faker.seed(42)
random.seed(42)

In [9]:
# Si 'fakeamos' los emails salen direcciones sin relación con el nombre o apellidos
# Lo solucionamos con una función que normalice el texto para luego usar nombre, apellidos e id + un dominio de correo fake en la dirección de email
import unicodedata

def clean_text(text: str) -> str:
    """Elimina tildes, convierte a minúsculas y quita espacios."""
    # Descompone caracteres con tilde en letra base + marca de acento
    nfkd = unicodedata.normalize('NFKD', text)
    # Filtra solo los caracteres base sin tildes ni caracteres no ASCII
    ascii_text = nfkd.encode('ASCII', 'ignore').decode('utf-8')
    return ascii_text.lower().replace(" ", "")

In [10]:
# Función para generar clientes

def generate_customers(n: int = 500) -> pd.DataFrame:
    channels = ["organic", "paid_search", "social_media", "email_marketing", "referral"]
    customers_data = []

    for i in range(1, n + 1):
        first_name = fake.first_name()
        last_name = fake.last_name()
        domain = fake.free_email_domain()
        
        # Correo coherente y único combinando el ID para evitar duplicados
        email = f"{clean_text(first_name)}.{clean_text(last_name)}{i}@{domain}"
        
        customers_data.append({
            "customer_number": i,
            "first_name": first_name,
            "last_name": last_name,
            "email": email,
            "country": "España",
            "city": fake.city(),
            "channel": random.choice(channels),
            "registration_date": fake.date_between(start_date="-2y", end_date="today")
        })

    return pd.DataFrame(customers_data)

# Generamos el dataframe

df_customers = generate_customers(500)
df_customers.head(10)

,customer_number,first_name,last_name,email,country,city,channel,registration_date
0,1,Feliciana,Cantón,feliciana.canton1@gmail.com,España,Ciudad,organic,2025-02-25
1,2,Emilio,Calatayud,emilio.calatayud2@hotmail.com,España,Palencia,referral,2025-07-29
2,3,Amílcar,Andrés,amilcar.andres3@gmail.com,España,Ceuta,organic,2025-10-02
3,4,Eligia,Amor,eligia.amor4@hotmail.com,España,Teruel,referral,2026-01-22
4,5,Judith,Rivas,judith.rivas5@gmail.com,España,Palencia,email_marketing,2025-04-09
5,6,Nydia,Agudo,nydia.agudo6@gmail.com,España,Guipúzcoa,organic,2025-04-08
6,7,Fabricio,Garrido,fabricio.garrido7@yahoo.com,España,Ávila,organic,2025-06-27
7,8,Cleto,Palomar,cleto.palomar8@yahoo.com,España,Zaragoza,organic,2024-10-08
8,9,Luna,Segovia,luna.segovia9@hotmail.com,España,Jaén,paid_search,2024-11-05
9,10,Carmina,Marcos,carmina.marcos10@hotmail.com,España,Ourense,paid_search,2025-02-01


In [4]:
# Función para generar categorías

def generate_categories() -> pd.DataFrame:
    categories_data = [
        {
            "id": 1,
            "category_name": "Smartphones",
            "description": "Teléfonos móviles inteligentes de última generación y terminales reacondicionados."
        },
        {
            "id": 2,
            "category_name": "Laptops",
            "description": "Ordenadores portátiles para productividad, gaming, desarrollo y diseño."
        },
        {
            "id": 3,
            "category_name": "Audio",
            "description": "Auriculares inalámbricos, altavoces Bluetooth y barras de sonido de alta fidelidad."
        },
        {
            "id": 4,
            "category_name": "Tablets & E-readers",
            "description": "Dispositivos táctiles multimedia y lectores de libros electrónicos."
        },
        {
            "id": 5,
            "category_name": "Wearables",
            "description": "Relojes inteligentes, pulseras de actividad y sensores biométricos."
        },
        {
            "id": 6,
            "category_name": "Componentes & Periféricos",
            "description": "Monitores, teclados mecánicos, ratones ergonómicos y almacenamiento externo."
        },
        {
            "id": 7,
            "category_name": "Smart Home",
            "description": "Dispositivos conectados, iluminación inteligente, enchufes y domótica para el hogar."
        }
    ]
    
    return pd.DataFrame(categories_data)

# Generamos el DataFrame

df_categories = generate_categories()
df_categories

,id,category_name,description
0,1,Smartphones,Teléfonos móviles inteligentes de última gener...
1,2,Laptops,"Ordenadores portátiles para productividad, gam..."
2,3,Audio,"Auriculares inalámbricos, altavoces Bluetooth ..."
3,4,Tablets & E-readers,Dispositivos táctiles multimedia y lectores de...
4,5,Wearables,"Relojes inteligentes, pulseras de actividad y ..."
5,6,Componentes & Periféricos,"Monitores, teclados mecánicos, ratones ergonóm..."
6,7,Smart Home,"Dispositivos conectados, iluminación inteligen..."


In [11]:
# Función para generar productos

def generate_products(target_count: int = 70) -> pd.DataFrame:
    # Catálogo de plantillas por categoría con rangos de precio realistas
    catalog_templates = {
        1: {  # Smartphones
            "base_names": ["iPhone 15", "Samsung Galaxy S24", "Xiaomi Redmi Note 13", "Google Pixel 8", "OnePlus 12"],
            "variants": ["Standard", "Pro", "Pro Max", "Ultra", "Lite"],
            "price_range": (180.0, 1400.0)
        },
        2: {  # Laptops
            "base_names": ["MacBook Air", "Lenovo ThinkPad", "Dell XPS", "Asus ROG Strix", "HP Envy"],
            "variants": ["13 inch", "15 inch", "Pro", "Gaming Edition", "UltraSlim"],
            "price_range": (550.0, 2400.0)
        },
        3: {  # Audio
            "base_names": ["Sony WH-1000XM5", "AirPods", "Bose QuietComfort", "JBL Flip", "Sennheiser Momentum"],
            "variants": ["Wireless", "Noise Cancelling", "Mini", "Sport", "Studio"],
            "price_range": (40.0, 380.0)
        },
        4: {  # Tablets & E-readers
            "base_names": ["iPad Air", "Samsung Galaxy Tab", "Kindle Paperwhite", "Lenovo Tab", "iPad Pro"],
            "variants": ["WiFi", "LTE", "Paper Edition", "Kids", "Max"],
            "price_range": (110.0, 950.0)
        },
        5: {  # Wearables
            "base_names": ["Apple Watch Series 9", "Garmin Fenix", "Xiaomi Smart Band", "Samsung Galaxy Watch", "Fitbit Charge"],
            "variants": ["GPS", "Cellular", "Active", "Classic", "Solar"],
            "price_range": (35.0, 650.0)
        },
        6: {  # Componentes & Periféricos
            "base_names": ["Logitech MX Master", "Monitor LG UltraGear 27", "Teclado Corsair K70", "SSD Kingston 1TB", "Webcam Logitech C920"],
            "variants": ["RGB", "Silent", "Mechanical", "4K", "Ergonomic"],
            "price_range": (25.0, 450.0)
        },
        7: {  # Smart Home
            "base_names": ["Bombilla Philips Hue", "Amazon Echo Dot", "Enchufe Inteligente TP-Link", "Cámara Ring", "Google Nest Hub"],
            "variants": ["Pack x2", "Gen 5", "Color", "Outdoor", "Mini"],
            "price_range": (15.0, 190.0)
        }
    }

    products_data = []
    generated_names = set()
    product_number = 1

    # Aseguramos un reparto equilibrado entre las 7 categorías
    categories_keys = list(catalog_templates.keys())

    while len(products_data) < target_count:
        cat_id = random.choice(categories_keys)
        cat_info = catalog_templates[cat_id]

        base = random.choice(cat_info["base_names"])
        variant = random.choice(cat_info["variants"])
        # Añadimos un código o capacidad para garantizar nombres únicos
        spec = random.choice(["64GB", "128GB", "256GB", "512GB", "V2", "Black", "Silver"])
        name = f"{base} {variant} ({spec})"

        if name in generated_names:
            continue
        generated_names.add(name)

        min_p, max_p = cat_info["price_range"]
        price = round(random.uniform(min_p, max_p), 2)
        
        # El coste representa entre el 65% y el 85% del PVP
        margin_factor = random.uniform(0.65, 0.85)
        cost = round(price * margin_factor, 2)

        products_data.append({
            "product_number": product_number,
            "product_name": name,
            "category_id": cat_id,
            "price": price,
            "cost": cost,
            "stock": random.randint(5, 120),
            "is_active": random.choices([True, False], weights=[0.9, 0.1])[0]  # 90% activos
        })
        product_number += 1

    return pd.DataFrame(products_data)

# Generamos los productos

df_products = generate_products(70)
df_products.head(10)

,product_number,product_name,category_id,price,cost,stock,is_active
0,1,Samsung Galaxy S24 Pro Max (128GB),1,1164.01,790.91,10,True
1,2,iPad Pro WiFi (512GB),4,458.13,355.49,29,True
2,3,Lenovo Tab Kids (128GB),4,233.96,184.25,119,True
3,4,Bombilla Philips Hue Outdoor (128GB),7,45.78,38.53,71,True
4,5,Garmin Fenix GPS (512GB),5,117.00,86.92,72,False
5,6,Xiaomi Smart Band Classic (V2),5,536.23,444.22,59,True
6,7,Samsung Galaxy Watch Cellular (Black),5,564.82,417.97,101,True
7,8,Teclado Corsair K70 Ergonomic (512GB),6,291.37,205.39,14,True
8,9,Dell XPS Pro (256GB),2,2202.10,1466.85,24,True
9,10,Monitor LG UltraGear 27 Silent (64GB),6,201.32,144.18,64,True


In [12]:
# Función para generar los pedidos

def generate_orders(df_customers: pd.DataFrame, target_orders: int = 2000) -> pd.DataFrame:
    # 1. Mapa de {customer_number: registration_date}
    # Aseguramos formato datetime para poder operar con fechas
    customer_reg_dates = {
        row["customer_number"]: pd.to_datetime(row["registration_date"])
        for _, row in df_customers.iterrows()
    }
    
    customer_ids = list(customer_reg_dates.keys())
    
    # Pesos realistas para los estados de pedido en un e-commerce
    statuses = ["delivered", "shipped", "confirmed", "pending", "cancelled", "returned"]
    status_weights = [0.70, 0.10, 0.08, 0.04, 0.05, 0.03]  # 70% ya entregados
    
    orders_data = []
    now = datetime.now()

    for order_id in range(1, target_orders + 1):
        cust_id = random.choice(customer_ids)
        reg_date = customer_reg_dates[cust_id]

        # Evitar errores si el cliente se registró hoy mismo
        if reg_date >= now:
            order_date = reg_date
        else:
            # Fecha del pedido entre su registro y el momento actual
            order_date = fake.date_time_between(start_date=reg_date, end_date=now)

        status = random.choices(statuses, weights=status_weights)[0]
        
        shipping_date = None
        delivery_date = None

        if status in ["shipped", "delivered", "returned"]:
            # Se envía entre 1 y 3 días después del pedido
            ship_candidate = order_date + timedelta(days=random.randint(1, 3), hours=random.randint(2, 10))
            if ship_candidate <= now:
                shipping_date = ship_candidate

        if status in ["delivered", "returned"] and shipping_date is not None:
            # Se entrega entre 1 y 4 días después del envío
            deliv_candidate = shipping_date + timedelta(days=random.randint(1, 4), hours=random.randint(2, 8))
            if deliv_candidate <= now:
                delivery_date = deliv_candidate
            else:
                # Si por cálculo caería en el futuro, ajustamos el estado a 'shipped'
                status = "shipped"

        # Dirección de envío creíble
        shipping_address = f"{fake.street_address()}, {fake.postcode()} {fake.city()}"

        orders_data.append({
            "order_number": order_id,
            "customer_number": cust_id,
            "order_status": status,
            "shipping_address": shipping_address,
            "order_date": order_date,
            "shipping_date": shipping_date,
            "delivery_date": delivery_date
        })

    return pd.DataFrame(orders_data)

# Generamos los 2000 pedidos

df_orders = generate_orders(df_customers, 2000)
df_orders.head()

,order_number,customer_number,order_status,shipping_address,order_date,shipping_date,delivery_date
0,1,19,delivered,"Cuesta Ramona Roura 29, 35613 Ceuta",2025-04-05 08:09:50,2025-04-06 14:09:50,2025-04-08 21:09:50
1,2,47,delivered,"Calle de Ámbar Benavente 18 Apt. 50 , 14576 Va...",2025-12-28 05:05:58,2025-12-29 08:05:58,2026-01-02 11:05:58
2,3,356,delivered,"Cañada Fermín Hernandez 668 Piso 2 , 46174 Málaga",2026-07-24 10:47:03,2026-07-25 12:47:03,2026-07-28 20:47:03
3,4,29,delivered,"Ronda Jesús Benitez 79, 51695 Albacete",2026-02-06 15:17:18,2026-02-08 23:17:18,2026-02-11 02:17:18
4,5,272,delivered,"Urbanización de Pelayo Perez 72 Apt. 94 , 1715...",2026-02-07 07:11:06,2026-02-10 11:11:06,2026-02-12 14:11:06


In [13]:
# Función para generar las líneas de pedido

def generate_order_items(df_orders: pd.DataFrame, df_products: pd.DataFrame) -> pd.DataFrame:
    # Diccionario de búsqueda: {product_number: price}
    product_prices = dict(zip(df_products["product_number"], df_products["price"]))
    all_product_ids = list(product_prices.keys())

    # Distribución de líneas por pedido para lograr una media de ~2.25 (~4500 líneas en total)
    items_per_order_choices = [1, 2, 3, 4]
    items_per_order_weights = [0.25, 0.35, 0.25, 0.15]

    order_items_data = []
    item_id = 1

    for order_number in df_orders["order_number"]:
        num_items = random.choices(items_per_order_choices, weights=items_per_order_weights)[0]
        
        # Seleccionamos productos distintos para este pedido
        selected_products = random.sample(all_product_ids, k=num_items)

        for prod_number in selected_products:
            base_price = product_prices[prod_number]
            quantity = random.choices([1, 2, 3], weights=[0.75, 0.20, 0.05])[0]

            # 20% de probabilidad de tener descuento promocional
            if random.random() < 0.20:
                # Descuento de entre el 5% y el 15% sobre el precio unitario
                discount_pct = random.uniform(0.05, 0.15)
                discount = round(base_price * discount_pct, 2)
            else:
                discount = 0.0

            order_items_data.append({
                "id": item_id,
                "order_number": order_number,
                "product_number": prod_number,
                "quantity": quantity,
                "buy_price": base_price,
                "discount": discount
            })
            item_id += 1

    return pd.DataFrame(order_items_data)

# Generamos las líneas de pedido

df_order_items = generate_order_items(df_orders, df_products)
print(f"Total líneas de pedido generadas: {len(df_order_items)}")
print(f"Media de productos por pedido: {len(df_order_items) / len(df_orders):.2f}")
df_order_items.head()

Total líneas de pedido generadas: 4616
Media de productos por pedido: 2.31


,id,order_number,product_number,quantity,buy_price,discount
0,1,1,69,2,136.22,0.0
1,2,1,14,1,603.70,0.0
2,3,1,49,1,274.15,0.0
3,4,1,33,1,1023.98,0.0
4,5,2,20,1,991.31,0.0


In [14]:
# Función para generar los pagos

def generate_payments(df_orders: pd.DataFrame, df_order_items: pd.DataFrame) -> pd.DataFrame:
    # 1. Calcular el total a pagar de cada pedido sumando sus líneas
    df_items_calc = df_order_items.copy()
    df_items_calc["line_total"] = (df_items_calc["buy_price"] - df_items_calc["discount"]) * df_items_calc["quantity"]
    
    order_totals = df_items_calc.groupby("order_number")["line_total"].sum().round(2).to_dict()

    # 2. Métodos de pago habituales
    payment_methods = ["card", "paypal", "bizum"]
    method_weights = [0.65, 0.20, 0.15]

    payments_data = []
    payment_id = 1

    for _, order in df_orders.iterrows():
        o_id = order["order_number"]
        o_status = order["order_status"]
        o_date = pd.to_datetime(order["order_date"])
        total_amount = order_totals.get(o_id, 0.0)

        # Determinar el estado del pago en coherencia con el estado del pedido
        if o_status in ["delivered", "shipped", "confirmed"]:
            pay_status = "completed"
        elif o_status == "pending":
            pay_status = "pending"
        elif o_status == "cancelled":
            # Un pedido cancelado puede haber fallado en el cobro o haber sido reembolsado
            pay_status = random.choice(["failed", "refunded"])
        elif o_status == "returned":
            pay_status = "refunded"
        else:
            pay_status = "completed"

        # Fecha de pago: unos minutos después de la orden (máximo 15 minutos de margen)
        payment_date = o_date + timedelta(minutes=random.randint(1, 15), seconds=random.randint(0, 59))

        payments_data.append({
            "id": payment_id,
            "order_number": o_id,
            "method": random.choices(payment_methods, weights=method_weights)[0],
            "payment_status": pay_status,
            "amount": total_amount,
            "payment_date": payment_date
        })
        payment_id += 1

    return pd.DataFrame(payments_data)

# Generamos los pagos

df_payments = generate_payments(df_orders, df_order_items)
print(f"Total pagos generados: {len(df_payments)}")
df_payments.head()

Total pagos generados: 2000


,id,order_number,method,payment_status,amount,payment_date
0,1,1,card,completed,2174.27,2025-04-05 08:17:16
1,2,2,card,completed,1590.54,2025-12-28 05:20:53
2,3,3,card,completed,2428.93,2026-07-24 10:52:09
3,4,4,card,completed,365.48,2026-02-06 15:22:16
4,5,5,card,completed,454.89,2026-02-07 07:20:49


In [15]:
# Función para generar las reviews

def generate_reviews(df_orders: pd.DataFrame, df_order_items: pd.DataFrame, review_rate: float = 0.35) -> pd.DataFrame:
    # 1. Identificar pedidos entregados
    delivered_order_ids = set(df_orders[df_orders["order_status"] == "delivered"]["order_number"])

    # 2. Filtrar order_items elegibles (que pertenezcan a pedidos entregados)
    eligible_items = df_order_items[df_order_items["order_number"].isin(delivered_order_ids)]["id"].tolist()

    # 3. Tomar una muestra aleatoria del ~35%
    sample_size = int(len(eligible_items) * review_rate)
    reviewed_item_ids = random.sample(eligible_items, k=sample_size)

    # Comentarios realistas por rango de puntuación
    comments_by_rating = {
        5: [
            "Excelente producto, superó mis expectativas.",
            "Llegó rapidísimo y en perfecto estado.",
            "Calidad brutal, totalmente recomendado.",
            "Funciona a la perfección, una compra 10/10.",
            "Recomendaría su compra a cualquiera.",
            "El mejor en su rango de precios. Ni una queja."
        ],
        4: [
            "Muy buen producto, aunque el embalaje venía algo golpeado.",
            "Cumple muy bien su función, buena relación calidad-precio.",
            "Satisfecho con la compra, repetiría.",
            "Recomendable por lo que pagas.",
            "Muy bueno, me gustan los acabados.",
            "Funciona a las mil maravillas."
        ],
        3: [
            "El producto está bien, pero esperaba un poco más de calidad.",
            "Correcto sin más. Cumple pero no destaca.",
            "Algo caro para lo que ofrece.",
            "Correcto para su rango de precios.",
            "Con este producto obtienes lo que pagas."
        ],
        2: [
            "No ha cumplido con lo prometido, bastante decepcionado.",
            "La batería dura menos de lo indicado en las especificaciones.",
            "Materiales de calidad mejorable.",
            "No me gusta, no lo volvería a comprar."
            "No lo recomendaría. No es lo que esperaba."
        ],
        1: [
            "Dejó de funcionar a los pocos días. Muy mala experiencia.",
            "No recomiendo este producto para nada.",
            "Pésima calidad, no lo volvería a comprar.",
            "Materiales de mala calidad y acabado pésimo."
        ]
    }

    # Distribución ponderada habitual de valoraciones en e-commerce
    rating_options = [1, 2, 3, 4, 5]
    rating_weights = [0.05, 0.08, 0.12, 0.30, 0.45]

    reviews_data = []
    review_id = 1

    for item_id in reviewed_item_ids:
        rating = random.choices(rating_options, weights=rating_weights)[0]
        
        # 60% de probabilidad de dejar comentario escrito; 40% solo deja estrellas (None)
        if random.random() < 0.60:
            comment = random.choice(comments_by_rating[rating])
        else:
            comment = None

        reviews_data.append({
            "id": review_id,
            "order_item_id": item_id,
            "rating": rating,
            "comment": comment
        })
        review_id += 1

    return pd.DataFrame(reviews_data)

# Generamos las reviews

df_reviews = generate_reviews(df_orders, df_order_items, review_rate=0.35)
print(f"Total reviews generadas: {len(df_reviews)}")
df_reviews.head(10)

Total reviews generadas: 1122


,id,order_item_id,rating,comment
0,1,4376,3,NaN
1,2,2290,4,Recomendable por lo que pagas.
2,3,3892,1,Dejó de funcionar a los pocos días. Muy mala e...
3,4,1491,5,NaN
4,5,2022,5,NaN
5,6,974,4,NaN
6,7,2165,5,"Funciona a la perfección, una compra 10/10."
7,8,2611,5,"Excelente producto, superó mis expectativas."
8,9,1274,4,Recomendable por lo que pagas.
9,10,1403,4,NaN


## Carga de datos en BigQuery

Se realiza la carga de los datos sintéticos en el dataset de BigQuery mediante la función load_data.

In [5]:
# Conexión a BigQuery
# Importar módulos
import os
from dotenv import load_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account
from google.api_core.exceptions import GoogleAPICallError

# Cargar variables de entorno
load_dotenv()
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Autenticación y cliente
credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH
)
client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

print(f"Conectado a BigQuery. Proyecto: {PROJECT_ID}")

Conectado a BigQuery. Proyecto: thebridge-ai-tc-sql


In [17]:
# Función para carga de datos

def load_data(name: str, dataframe: pd.DataFrame) -> bool:
    # Carga un DataFrame en una tabla de BigQuery, captura posibles errores y valida el número de filas cargadas en el servidor
    table_ref = f"{PROJECT_ID}.{DATASET_ID}.{name}"
    expected_rows = len(dataframe)
        
    # Configurar la carga
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",  # Sobreescribe
    )

    # Cargar datos
    try:
        # Iniciar y esperar el trabajo de carga
        job = client.load_table_from_dataframe(dataframe, table_ref, job_config=job_config)
        job.result()  # Esperar a que termine
        # Verificar si el job reportó errores internos
        if job.errors:
            print(f"[ERROR] Errores reportados en la carga de '{name}': {job.errors}")
            return False
        # Validación en el servidor
        table = client.get_table(table_ref)
        actual_rows = table.num_rows
        if actual_rows == expected_rows:
            print(f"[OK] '{name}': {actual_rows} filas validadas en BigQuery (coincide con el DataFrame).")
            return True
        else:
            print(f"[ADVERTENCIA] '{name}': Se esperaban {expected_rows} filas pero BigQuery reporta {actual_rows}.")
            return False
    except GoogleAPICallError as e:
        print(f"[ERROR API] Fallo al cargar '{name}' en BigQuery: {e}")
        return False
    except Exception as e:
        print(f"[ERROR INESPERADO] Error procesando '{name}': {e}")
        return False

In [19]:
# Carga de datos en las diferentes tablas

tables_to_load = [
    ("categories", df_categories),
    ("customers", df_customers),
    ("products", df_products),
    ("orders", df_orders),
    ("order_items", df_order_items),
    ("payments", df_payments),
    ("reviews", df_reviews),
]

for table_name, df in tables_to_load:
    success = load_data(table_name, df)
    if not success:
        print(f"Carga abortada debido a un fallo en la tabla {table_name}")
        break

c:\Users\Rubén\Documents\01-Estudios\Bootcamp IA\00 - Repos\tc-sql-Ruben-Jimenez-Gutierrez\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[OK] 'categories': 7 filas validadas en BigQuery (coincide con el DataFrame).


c:\Users\Rubén\Documents\01-Estudios\Bootcamp IA\00 - Repos\tc-sql-Ruben-Jimenez-Gutierrez\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[OK] 'customers': 500 filas validadas en BigQuery (coincide con el DataFrame).


c:\Users\Rubén\Documents\01-Estudios\Bootcamp IA\00 - Repos\tc-sql-Ruben-Jimenez-Gutierrez\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


...

[OK] 'order_items': 4616 filas validadas en BigQuery (coincide con el DataFrame).


c:\Users\Rubén\Documents\01-Estudios\Bootcamp IA\00 - Repos\tc-sql-Ruben-Jimenez-Gutierrez\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[OK] 'payments': 2000 filas validadas en BigQuery (coincide con el DataFrame).


c:\Users\Rubén\Documents\01-Estudios\Bootcamp IA\00 - Repos\tc-sql-Ruben-Jimenez-Gutierrez\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[OK] 'reviews': 1122 filas validadas en BigQuery (coincide con el DataFrame).
